# EDA - Cleaning and Procesing

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('src'))

# DI
from di.container import AppContainer

load_dotenv()

True

## DI Init

In [2]:
container = AppContainer()
container.config.from_dict({
    "aws": {
        "s3_bucket_name": os.getenv("AWS_S3_BUCKET_NAME"),
        "region_name": os.getenv("AWS_REGION_NAME")
    },
    "db": {
        "connection_string": os.getenv("DB_CONNECTION_STRING")
    }
})

rds_to_pandas_usecase = container.rds_to_pandas_usecase()

## Fetch Data from AWS RDS

In [3]:
# BTC-USD ETH-USD SOL-USD BNB-USD DOGE-USD"
TICKER = "BTC-USD" 
TABLE_NAME = os.getenv("AWS_RDS_TABLE_NAME", "crypto_market_data")

df = rds_to_pandas_usecase.execute(ticker=TICKER, table_name=TABLE_NAME)
print(f"Fetched {len(df)} records for {TICKER}.")
df.head()

2026-04-04 23:34:47,482 | INFO     | [App] Starting RDS -> Pandas retrieval for ticker: BTC-USD
2026-04-04 23:34:55,768 | INFO     | [App] Successfully loaded 820 records into Pandas DataFrame

--- Data for BTC-USD ---
   id   ticker timestamp_utc    open_price    high_price     low_price  \
0   2  BTC-USD    2024-01-01  42280.234375  44175.437500  42214.976562   
1  73  BTC-USD    2024-01-02  44187.140625  45899.707031  44176.949219   
2  75  BTC-USD    2024-01-03  44961.601562  45503.242188  40813.535156   
3  77  BTC-USD    2024-01-04  42855.816406  44770.023438  42675.175781   
4  79  BTC-USD    2024-01-05  44192.980469  44353.285156  42784.718750   

    close_price       volume         source  
0  44167.332031  18426978443  yahoo_finance  
1  44957.968750  39335274536  yahoo_finance  
2  42848.175781  46342323118  yahoo_finance  
3  44179.921875  30448091210  yahoo_finance  
4  44162.691406  32336029347  yahoo_finance  
Fetched 820 records for BTC-USD.


,id,ticker,timestamp_utc,open_price,high_price,low_price,close_price,volume,source
0,2,BTC-USD,2024-01-01,42280.234375,44175.437500,42214.976562,44167.332031,18426978443,yahoo_finance
1,73,BTC-USD,2024-01-02,44187.140625,45899.707031,44176.949219,44957.968750,39335274536,yahoo_finance
2,75,BTC-USD,2024-01-03,44961.601562,45503.242188,40813.535156,42848.175781,46342323118,yahoo_finance
3,77,BTC-USD,2024-01-04,42855.816406,44770.023438,42675.175781,44179.921875,30448091210,yahoo_finance
4,79,BTC-USD,2024-01-05,44192.980469,44353.285156,42784.718750,44162.691406,32336029347,yahoo_finance


## Data Cleaning

### Rename Columns and Drop Not need

In [4]:
df = df.drop(columns=['source'])

rename_to = {
    'timestamp_utc': 'date',
    'open_price': 'open',
    'high_price': 'high',
    'low_price': 'low',
    'close_price': 'close'
}
df = df.rename(columns=rename_to)
df.head()

,id,ticker,date,open,high,low,close,volume
0,2,BTC-USD,2024-01-01,42280.234375,44175.437500,42214.976562,44167.332031,18426978443
1,73,BTC-USD,2024-01-02,44187.140625,45899.707031,44176.949219,44957.968750,39335274536
2,75,BTC-USD,2024-01-03,44961.601562,45503.242188,40813.535156,42848.175781,46342323118
3,77,BTC-USD,2024-01-04,42855.816406,44770.023438,42675.175781,44179.921875,30448091210
4,79,BTC-USD,2024-01-05,44192.980469,44353.285156,42784.718750,44162.691406,32336029347


### Handle Missing Values

In [5]:
missing_map = df.isnull()

# sum of columns -> final column total
missing_total = missing_map.sum().sum()
print(f"Initial missing values: {missing_total}")

if missing_total > 0:
    df = df.fillna(method='ffill')
    print(f"Total Missing after fill: {df.isnull().sum().sum()}")
else :
    print("No Missing")

Initial missing values: 0
No Missing


### Convert Data Types

In [6]:
df.dtypes

id                 int64
ticker            object
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume            object
dtype: object

Convert Volume Column to Numberic

In [ ]:
df['volume'] = pd.to_numeric(df['volume'])
df.dtypes

id                 int64
ticker            object
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object

## 4. Line-Chart of Bitcoin

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
plt.plot(df['timestamp_utc'],
 df['close'], label='price')
plt.title(f"{TICKER} Price")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.show()

KeyError: 'timestamp_utc'

<Figure size 1200x600 with 0 Axes>